In [4]:
import torch 
import triton 
import triton.language as tl
from triton.runtime import driver 

DEVICE = triton.runtime.driver.active.get_active_torch_device()


def is_hip():
    return triton.runtime.driver.active.get_current_target().backend == "hip"


def is_cdna():
    return is_hip() and triton.runtime.driver.active.get_current_target().arch in ('gfx940', 'gfx941', 'gfx942',
                                                                                   'gfx90a')


In [ ]:
def naive_softmax(x):

    x_max = x.max(dim=1)[0]

    z = x- x_max[:,None]

    numerator = torch.exp(z)
    denominator = x.sum(dim=1)

    z = numerator/denominator[:,None]

    return z


In [ ]:
@triton.jit
def softmax_kernel(output_ptr,
                   input_ptr,
                   output_row_stride,
                   input_row_stride,
                   n_row,
                   n_col,
                   num_stages: tl.constexpr
                ):
    row_start = tl.program_id(0)
    row_step = tl.num_programs(0)

    for row_idx in tl.range(row_start,n_row,step=row_step,num_stages=num_stages):
        row_start_ptr = input_ptr + row_idx*input_row_stride

        col_offest  = tl.arange(0,BLOCK_SIZE) 
        input_ptrs = row_start_ptr + col_offest
        mask = col_offest<n_col

        row = tl.load(input_ptr,mask=mask,other=-float('inf'))
        
        row_max = tl.max(input_ptr,axis=0)

        row_max = row - row_max

        numerator = tl.exp(row_max)
        denominator = tl.sum(row,axis=0)

        output = numerator/denominator

        output_row_start_ptr = output_ptr+row_idx*output_row_stride
        output_ptr = output_ptr+col_offest

        tl.store(output_ptr,output,mask=mask)
